# Fill long-read ancestry gaps (`lr_PC` kNN)

Backfill missing `ancestry_pred` / `ancestry_pred_other` (and `population` when
needed) for long-read samples so within-population PCA
(`tractor_07_pca_within_population.ipynb`) can include the full joint callset.

## Prerequisites

1. `tractor_00_cov_rebuild_source.ipynb` → `tractor_mix/covariates.source_rebuilt.csv.gz`
2. Global long-read PCs available (from `tractor_05` / `tractor_mix/pca/deepvariant_lr_v2/global_pcs.tsv`)
3. Controls + `lr_PC*` merged into covariates:

```bash
python3 scripts/merge_lr_global_pcs_into_covariates.py \
  --global-pcs tractor_mix/pca/deepvariant_lr_v2/global_pcs.tsv \
  --control-metadata tractor_mix/reference_controls/control_sample_metadata.tsv
```

## Fill rules

1. **From `population`** — if continental `population` is present but ancestry is
   missing, copy into `ancestry_pred_other` (lowercase). Copy into hard
   `ancestry_pred` only when the label is not `oth` (`OTH` never becomes hard
   `ancestry_pred`).
2. **From `lr_PC` kNN** — if both ancestry and `population` are missing but
   `has_lr_pcs`, infer soft (`ancestry_pred_other` / `population`) and hard
   (`ancestry_pred`) labels from `lr_PC1`–`lr_PC10` (nearest centroid + kNN,
   k=15). Hard votes use hard-labeled neighbors only (never `oth`).
3. **Hard `ancestry_pred` remediation** — if AoU `has_lr_pcs` and
   `ancestry_pred` is missing or `oth`, overwrite hard pred only from hard
   neighbors; leave `ancestry_pred_other` / `population` unchanged.

Does **not** overwrite existing hard `ancestry_pred`, soft other, or
`population`, except rule 3’s soft-`oth` → hard remediation. See
`tractor_mix/reference_controls/COVARIATE_FILLS.md`.

## Outputs

- Updated `tractor_mix/covariates.source_rebuilt.csv.gz` (in place by default)
- Audit TSV: `tractor_mix/reference_controls/lr_ancestry_knn_fills.tsv`
- Optional upload to `$WORKSPACE_BUCKET/covariates/`

Set `FILL_DRY_RUN=true` to compute and plot without writing.
Set `FILL_HIGH_CONFIDENCE_ONLY=true` to skip low-confidence kNN votes.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import subprocess
import sys

for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError(
            "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
            "Upload scripts/ to gs://WORKSPACE/scripts/."
        )
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        ["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"]
    )
    sys.path.insert(0, str(_dest))

from terra_notebook import init_notebook

SCRIPTS = init_notebook("fill_lr_ancestry_from_pcs.py", "workspace_paths.py")
from workspace_paths import data_root
from fill_lr_ancestry_from_pcs import (
    APPLIED_METHODS,
    ancestry_gap_mask,
    fill_lr_ancestry,
    is_long_read,
    is_missing,
    summarize_gaps,
)

ROOT = data_root()
print("ROOT:", ROOT)
print("scripts:", sorted(p.name for p in SCRIPTS.glob("*")))


In [ ]:
from pathlib import Path
import os
import subprocess

import matplotlib.pyplot as plt
import pandas as pd

try:
    display
except NameError:
    def display(value):
        print(value)

# ROOT is tractor_mix/ locally (see workspace_paths.data_root).
DEFAULT_COV = ROOT / "covariates.source_rebuilt.csv.gz"
DEFAULT_AUDIT = ROOT / "reference_controls" / "lr_ancestry_knn_fills.tsv"

COV_PATH = Path(os.environ.get("FILL_COV_PATH", os.environ.get("TRACTOR_COVARIATES_GCS", str(DEFAULT_COV))))
OUT_COV_PATH = Path(os.environ.get("FILL_OUT_COV_PATH", str(DEFAULT_COV)))
OUT_AUDIT_PATH = Path(os.environ.get("FILL_OUT_AUDIT_PATH", str(DEFAULT_AUDIT)))
DRY_RUN = os.environ.get("FILL_DRY_RUN", "false").lower() in {"1", "true", "yes"}
HIGH_CONFIDENCE_ONLY = os.environ.get("FILL_HIGH_CONFIDENCE_ONLY", "false").lower() in {
    "1",
    "true",
    "yes",
}
UPLOAD = os.environ.get("FILL_UPLOAD", "true").lower() in {"1", "true", "yes"}

if str(COV_PATH).startswith("gs://"):
    print(f"Fetching {COV_PATH} → {DEFAULT_COV}")
    DEFAULT_COV.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["gsutil", "cp", str(COV_PATH), str(DEFAULT_COV)])
    COV_PATH = DEFAULT_COV

print("COV_PATH:", COV_PATH)
print("OUT_COV_PATH:", OUT_COV_PATH)
print("OUT_AUDIT_PATH:", OUT_AUDIT_PATH)
print("DRY_RUN:", DRY_RUN)
print("HIGH_CONFIDENCE_ONLY:", HIGH_CONFIDENCE_ONLY)
print("UPLOAD:", UPLOAD)


## 1. Load covariates and summarize ancestry gaps

Long-read here means any of: `has_lr_pcs`, `has_lr_tech`, `lr_meet_qc`,
`final_releasable_v9`, non-null `lr_phase`, or `is_reference_control`.


In [ ]:
cov = pd.read_csv(COV_PATH, dtype={"research_id": str}, low_memory=False)
assert "research_id" in cov.columns
assert "ancestry_pred_other" in cov.columns

before = summarize_gaps(cov)
print("Before fill:")
for k, v in before.items():
    print(f"  {k}: {v:,}")

lr = is_long_read(cov)
miss = ancestry_gap_mask(cov)
gap_lr = cov.loc[lr & miss].copy()
print(f"\nLong-read ancestry gaps: {len(gap_lr):,}")
display(
    pd.DataFrame(
        {
            "has_lr_pcs": [(gap_lr["has_lr_pcs"] == True).sum()],
            "has_population": [(~is_missing(gap_lr["population"])).sum()],
            "final_releasable_v9": [(gap_lr["final_releasable_v9"] == True).sum()],
            "is_reference_control": [(gap_lr.get("is_reference_control", False) == True).sum()],
        }
    )
)


## 2. Apply fills

Uses `scripts/fill_lr_ancestry_from_pcs.py`. Review the audit table before
uploading if running outside dry-run.


In [ ]:
filled, audit = fill_lr_ancestry(
    cov,
    fill_low_confidence=not HIGH_CONFIDENCE_ONLY,
)
after = summarize_gaps(filled)
applied = audit.loc[audit["method"].isin(APPLIED_METHODS)] if len(audit) else audit
skipped = (
    audit.loc[audit["method"] == "lr_pc_knn_skipped_low_conf"] if len(audit) else audit
)

print(f"LR ancestry gaps: {before['lr_missing_ancestry']:,} → {after['lr_missing_ancestry']:,}")
print(f"Fills applied: {len(applied):,}")
if len(applied):
    print("\nBy method:")
    display(applied["method"].value_counts().rename("n").to_frame())
    print("By suggested label:")
    display(applied["suggested"].value_counts().rename("n").to_frame())
if len(skipped):
    print(f"\nLow-confidence kNN skipped: {len(skipped):,}")
    display(skipped[["research_id", "suggested", "centroid", "knn15_frac", "centroid_margin"]])

print("\nAudit preview:")
display(audit.head(20) if len(audit) else audit)


## 3. QC: kNN assignments in `lr_PC1` vs `lr_PC2`

Background = labeled joint-callset samples by `ancestry_pred_other`. Overlay =
rows filled by `lr_pc_knn` (high-confidence colored; low-confidence black with
yellow edge).


In [ ]:
POP_COLORS = {
    "afr": "#e41a1c",
    "amr": "#ff7f00",
    "eas": "#4daf4a",
    "eur": "#377eb8",
    "sas": "#984ea3",
    "mid": "#a65628",
    "oth": "#999999",
}
ANC_ORDER = ["afr", "amr", "eas", "eur", "sas", "mid", "oth"]

ref = filled.loc[
    (filled["has_lr_pcs"] == True) & ~is_missing(filled["ancestry_pred_other"])
].copy()
ref["label"] = ref["ancestry_pred_other"].astype(str).str.lower()
knn_rows = applied.loc[applied["method"] == "lr_pc_knn"].copy() if len(applied) else applied

fig, ax = plt.subplots(figsize=(8.5, 7.5))
for lab in ANC_ORDER:
    sub = ref.loc[ref["label"] == lab]
    if sub.empty:
        continue
    ax.scatter(
        sub["lr_PC1"],
        sub["lr_PC2"],
        s=6,
        alpha=0.10,
        c=POP_COLORS[lab],
        linewidths=0,
        label=f"{lab} (n={len(sub):,})",
        zorder=1,
    )

if len(knn_rows):
    for _, row in knn_rows.iterrows():
        if bool(row.get("high_confidence", False)):
            color = POP_COLORS.get(str(row["suggested"]), "#333333")
            edge = "black"
        else:
            color = "black"
            edge = "yellow"
        ax.scatter(
            row["lr_PC1"],
            row["lr_PC2"],
            s=70,
            alpha=0.95,
            c=color,
            edgecolors=edge,
            linewidths=0.8,
            zorder=3,
        )
    ax.scatter([], [], s=70, c="gray", edgecolors="black", label="kNN high-conf")
    ax.scatter([], [], s=70, c="black", edgecolors="yellow", label="kNN low-conf")

ax.set_xlabel("lr_PC1")
ax.set_ylabel("lr_PC2")
ax.set_title("kNN ancestry fills over labeled long-read PC space")
ax.legend(frameon=False, loc="best", fontsize=8, markerscale=1.2)
ax.set_aspect("equal", adjustable="datalim")
fig.tight_layout()
plt.show()

print(
    "has_lr_pcs missing ancestry after fill:",
    after["has_lr_pcs_missing_ancestry"],
)
print(
    "final_releasable missing ancestry after fill:",
    after["final_releasable_missing_ancestry"],
)


## 4. Write outputs (and optional bucket upload)

Remaining long-read gaps after fill are typically withdrawn / non-releasable /
Phase-1 leftovers with no `lr_PC*` and no `population` — safe to ignore for
within-pop PCA.


In [ ]:
if DRY_RUN:
    print("DRY_RUN=true — not writing covariates or audit")
else:
    OUT_AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
    OUT_COV_PATH.parent.mkdir(parents=True, exist_ok=True)
    audit.to_csv(OUT_AUDIT_PATH, sep="\t", index=False)
    filled.to_csv(OUT_COV_PATH, index=False, compression="gzip")
    print("wrote", OUT_COV_PATH)
    print("wrote", OUT_AUDIT_PATH)

ws = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
if not DRY_RUN and UPLOAD and ws:
    dest = f"{ws}/covariates/{OUT_COV_PATH.name}"
    audit_dest = f"{ws}/covariates/{OUT_AUDIT_PATH.name}"
    subprocess.check_call(["gsutil", "cp", str(OUT_COV_PATH), dest])
    subprocess.check_call(["gsutil", "cp", str(OUT_AUDIT_PATH), audit_dest])
    print("uploaded", dest)
    print("uploaded", audit_dest)
elif not ws:
    print("WORKSPACE_BUCKET unset; skip upload")
else:
    print("upload skipped")

print("\nNext: run tractor_07_pca_within_population.ipynb with PCA_COV_URI pointing at the filled covariates.")
